
# NUS ST3247 Simulation — Tutor-Style Case Study
## Capital Bikeshare Demand: from real event counts to Monte Carlo, NHPP simulation, variance reduction, MCMC, and discrete-event simulation

### What this notebook is trying to teach

This notebook is deliberately **not** a standard predictive-ML notebook.  
The goal is to think like a student of **ST3247 Simulation**:

> **Start with an observed stochastic system → propose a probabilistic data-generating mechanism → simulate from it → validate it → quantify uncertainty → improve simulation efficiency → use simulation for decisions.**

We use the **Capital Bikeshare hourly dataset** from Washington, D.C. (2011–2012). The data contain hourly rental counts together with calendar and weather information.

**Online sources**
- UCI Machine Learning Repository — Bike Sharing Dataset  
  `https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset`
- Kaggle mirror — Bike Sharing in Washington D.C.  
  `https://www.kaggle.com/datasets/marklvl/bike-sharing-dataset`

The important variable is:
$$
\
Y_t = \text{number of rentals observed in hour }t.
\
$$
Because $(Y_t)$ is an **event count**, it gives us a natural path into Poisson models, non-homogeneous Poisson processes, Monte Carlo experiments, simulation-based capacity planning, and Bayesian simulation.

---

## Learning objectives

By the end, you should be able to explain and implement:

1. pseudo-random number generation and why simulation begins with \(U(0,1)\);
2. inverse-transform sampling for continuous random variables;
3. discrete random-variable simulation;
4. Poisson count modelling and overdispersion diagnostics;
5. piecewise non-homogeneous Poisson-process simulation;
6. Monte Carlo estimation and Monte Carlo standard error;
7. convergence at the $(O(n^{-1/2}))$ rate;
8. variance reduction using a **control variate**;
9. bootstrap uncertainty versus Monte Carlo uncertainty;
10. discrete-event queue simulation;
11. Metropolis–Hastings MCMC for a Poisson rate;
12. computational complexity and vectorisation trade-offs.

Throughout the notebook, each section contains:

- **Question**
- **Model / algorithm**
- **Implementation**
- **Visual diagnostic**
- **Inference**
- **Caveat**

That structure is useful far beyond this module.



## 0. Environment setup

The notebook uses **NumPy, pandas, SciPy, and Bokeh**.  
Bokeh is used for interactive plots.

If a package is missing, uncomment the installation line below.


In [1]:
# Uncomment only if your environment is missing packages.
# %pip install -q numpy pandas scipy bokeh requests

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from io import BytesIO
import heapq
import math
import zipfile
import warnings

import numpy as np
import pandas as pd

from scipy import stats
from scipy.special import gammaln

from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import Band, ColumnDataSource, HoverTool
from bokeh.plotting import figure

output_notebook()

RANDOM_SEED = 3247
rng = np.random.default_rng(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


Loading BokehJS ...


# Part I — Data acquisition and statistical framing

## 1. Why this dataset is suitable for a simulation course

A predictive course might ask:
$$
\
\hat y = f(\text{hour}, \text{weather}, \text{temperature}, \ldots).
\
$$
ST3247 asks a different question:
$$
\
\textbf{What stochastic mechanism could plausibly have produced }Y_t?
\
$$
For example:
$$
\
Y_t \sim \operatorname{Poisson}(\lambda_t)
\
$$
with an intensity $(\lambda_t)$ that changes by hour.

This turns the observed dataset into a **calibration dataset** for a simulator.

### Important modelling distinction

The observed count `cnt` is the **system-wide number of rentals in an hour**.  
Later, when we build a queueing example, we will explicitly create a hypothetical station that receives only a fraction of that demand. That station-level scenario is an educational *what-if*, not a claim that the original dataset contains station queues.


In [2]:

@dataclass
class BikeSharingLoader:
    """Robust loader: local/Kaggle paths -> UCI download -> synthetic offline fallback."""

    uci_zip_url: str = (
        "https://archive.ics.uci.edu/static/public/275/"
        "bike+sharing+dataset.zip"
    )

    def _candidate_paths(self) -> list[Path]:
        return [
            Path("hour.csv"),
            Path("./data/hour.csv"),
            Path("/kaggle/input/bike-sharing-dataset/hour.csv"),
            Path("/kaggle/input/bike-sharing-in-washington-dc-dataset/hour.csv"),
        ]

    def _load_local(self) -> tuple[pd.DataFrame, str] | None:
        for path in self._candidate_paths():
            if path.exists():
                return pd.read_csv(path), f"local file: {path}"
        return None

    def _load_uci(self) -> tuple[pd.DataFrame, str]:
        import requests

        response = requests.get(self.uci_zip_url, timeout=30)
        response.raise_for_status()

        with zipfile.ZipFile(BytesIO(response.content)) as zf:
            with zf.open("hour.csv") as fh:
                df = pd.read_csv(fh)

        return df, "UCI Bike Sharing Dataset"

    @staticmethod
    def _synthetic_fallback(seed: int = RANDOM_SEED) -> tuple[pd.DataFrame, str]:
        """
        Offline-only fallback with the SAME schema.
        It is intentionally realistic enough to let every teaching cell execute,
        but it is NOT a substitute for the real dataset.
        """
        local_rng = np.random.default_rng(seed)

        dates = pd.date_range("2011-01-01", "2012-12-31 23:00:00", freq="h")
        n = len(dates)

        hr = dates.hour.to_numpy()
        weekday = dates.dayofweek.to_numpy()
        workingday = (weekday < 5).astype(int)
        mnth = dates.month.to_numpy()
        yr = (dates.year - 2011).to_numpy()

        morning_peak = 145 * np.exp(-0.5 * ((hr - 8) / 1.7) ** 2)
        evening_peak = 190 * np.exp(-0.5 * ((hr - 17.5) / 2.2) ** 2)
        midday = 70 * np.exp(-0.5 * ((hr - 13) / 4.5) ** 2)
        weekend_shift = (1 - workingday) * (
            85 * np.exp(-0.5 * ((hr - 14) / 4.0) ** 2)
        )
        trend = 1 + 0.45 * yr
        seasonality = 1 + 0.28 * np.sin(2 * np.pi * (mnth - 3) / 12)

        lam = (
            8
            + workingday * (morning_peak + evening_peak + 0.4 * midday)
            + (1 - workingday) * (weekend_shift + 0.7 * midday)
        ) * trend * seasonality
        lam = np.clip(lam, 1.0, None)

        # Gamma-Poisson mixture to create realistic overdispersion.
        shape = 8.0
        gamma_mult = local_rng.gamma(shape=shape, scale=1/shape, size=n)
        cnt = local_rng.poisson(lam * gamma_mult)

        temp = np.clip(
            0.50
            + 0.32 * np.sin(2*np.pi*(mnth-4)/12)
            + local_rng.normal(0, 0.08, n),
            0, 1,
        )
        hum = np.clip(local_rng.normal(0.62, 0.15, n), 0, 1)
        windspeed = np.clip(local_rng.gamma(2.0, 0.08, n), 0, 1)

        weather_probs = np.column_stack([
            0.68 - 0.15 * hum,
            0.24 + 0.08 * hum,
            0.075 + 0.06 * hum,
            np.full(n, 0.005),
        ])
        weather_probs = weather_probs / weather_probs.sum(axis=1, keepdims=True)
        u = local_rng.random(n)
        cum = np.cumsum(weather_probs, axis=1)
        weathersit = (u[:, None] > cum).sum(axis=1) + 1

        casual_share = np.clip(
            0.18 + 0.18 * (1-workingday) + 0.12 * temp,
            0.08, 0.55,
        )
        casual = local_rng.binomial(cnt, casual_share)
        registered = cnt - casual

        season = np.select(
            [np.isin(mnth, [12,1,2]), np.isin(mnth, [3,4,5]),
             np.isin(mnth, [6,7,8])],
            [1,2,3],
            default=4
        )

        df = pd.DataFrame({
            "instant": np.arange(1, n+1),
            "dteday": dates.date.astype(str),
            "season": season,
            "yr": yr,
            "mnth": mnth,
            "hr": hr,
            "holiday": 0,
            "weekday": (weekday + 1) % 7,
            "workingday": workingday,
            "weathersit": weathersit,
            "temp": temp,
            "atemp": temp,
            "hum": hum,
            "windspeed": windspeed,
            "casual": casual,
            "registered": registered,
            "cnt": cnt,
        })
        return df, "SYNTHETIC OFFLINE FALLBACK"

    def load(self) -> tuple[pd.DataFrame, str]:
        local = self._load_local()
        if local is not None:
            return local

        try:
            return self._load_uci()
        except Exception as exc:
            warnings.warn(
                "Real online data could not be downloaded. "
                f"Using synthetic fallback so the notebook remains runnable. "
                f"Original error: {exc}"
            )
            return self._synthetic_fallback()


loader = BikeSharingLoader()
df, DATA_SOURCE = loader.load()

print("DATA SOURCE:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())


DATA SOURCE: local file: data/hour.csv
Shape: (17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.2400,0.2879,0.8100,0.0000,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.2200,0.2727,0.8000,0.0000,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.2200,0.2727,0.8000,0.0000,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.2400,0.2879,0.7500,0.0000,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.2400,0.2879,0.7500,0.0000,0,1,1



### Interpretation checkpoint

For the real UCI data you should see roughly **17k hourly rows**.

Before any simulation, always answer:

1. What is the stochastic unit? → one hour.
2. What is the random variable? → number of rentals in that hour.
3. What conditioning information exists? → hour, working day, weather, season, etc.
4. What do we want the simulator to reproduce? → the distribution and time pattern of counts.

If the notebook prints `SYNTHETIC OFFLINE FALLBACK`, connect to the internet or place `hour.csv` beside the notebook before drawing empirical conclusions.


In [3]:

df = df.copy()

df["date"] = pd.to_datetime(df["dteday"])
df["datetime"] = df["date"] + pd.to_timedelta(df["hr"], unit="h")

weather_map = {
    1: "Clear/Partly Cloudy",
    2: "Mist/Cloudy",
    3: "Light Rain/Snow",
    4: "Heavy Rain/Snow/Fog",
}
df["weather_label"] = df["weathersit"].map(weather_map)

df["day_type"] = np.where(df["workingday"].eq(1), "Working day", "Non-working day")

print(df.info())
display(df.describe(include="all").T)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   instant        17379 non-null  int64         
 1   dteday         17379 non-null  object        
 2   season         17379 non-null  int64         
 3   yr             17379 non-null  int64         
 4   mnth           17379 non-null  int64         
 5   hr             17379 non-null  int64         
 6   holiday        17379 non-null  int64         
 7   weekday        17379 non-null  int64         
 8   workingday     17379 non-null  int64         
 9   weathersit     17379 non-null  int64         
 10  temp           17379 non-null  float64       
 11  atemp          17379 non-null  float64       
 12  hum            17379 non-null  float64       
 13  windspeed      17379 non-null  float64       
 14  casual         17379 non-null  int64         
 15  registered     1737

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
instant,"17,379.0000",NaN,NaN,NaN,"8,690.0000",1.0000,"4,345.5000","8,690.0000","13,034.5000","17,379.0000","5,017.0295"
dteday,17379,731,2012-12-31,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
season,"17,379.0000",NaN,NaN,NaN,2.5016,1.0000,2.0000,3.0000,3.0000,4.0000,1.1069
yr,"17,379.0000",NaN,NaN,NaN,0.5026,0.0000,0.0000,1.0000,1.0000,1.0000,0.5000
mnth,"17,379.0000",NaN,NaN,NaN,6.5378,1.0000,4.0000,7.0000,10.0000,12.0000,3.4388
hr,"17,379.0000",NaN,NaN,NaN,11.5468,0.0000,6.0000,12.0000,18.0000,23.0000,6.9144
holiday,"17,379.0000",NaN,NaN,NaN,0.0288,0.0000,0.0000,0.0000,0.0000,1.0000,0.1672
weekday,"17,379.0000",NaN,NaN,NaN,3.0037,0.0000,1.0000,3.0000,5.0000,6.0000,2.0058
workingday,"17,379.0000",NaN,NaN,NaN,0.6827,0.0000,0.0000,1.0000,1.0000,1.0000,0.4654
weathersit,"17,379.0000",NaN,NaN,NaN,1.4253,1.0000,1.0000,1.0000,2.0000,4.0000,0.6394



## 2. Basic data-quality checks

Simulation is particularly sensitive to bad inputs because a simulator can **amplify** incorrect assumptions.

We therefore check:

- missingness;
- duplicate timestamps;
- impossible negative counts;
- whether `casual + registered == cnt`.

That final equality is a useful reconciliation control.


In [4]:

quality = pd.Series({
    "rows": len(df),
    "missing_cells": int(df.isna().sum().sum()),
    "duplicate_datetimes": int(df["datetime"].duplicated().sum()),
    "negative_counts": int((df["cnt"] < 0).sum()),
    "count_reconciliation_failures": int(
        ((df["casual"] + df["registered"]) != df["cnt"]).sum()
    ),
})

display(quality.to_frame("value"))


,value
rows,17379
missing_cells,0
duplicate_datetimes,0
negative_counts,0
count_reconciliation_failures,0



**Inference rule:** if the reconciliation check fails on the real dataset, do not silently continue.  
First determine whether the issue comes from parsing, a modified copy of the data, or the source itself.



# Part II — Exploratory analysis as simulator diagnostics

## 3. Does demand behave as if it had a constant rate?

A homogeneous Poisson model assumes one constant rate $(\lambda)$.

But bike demand obviously changes throughout the day.  
We first visualize the conditional mean:
$$
\
\hat\lambda_h
=
\frac{1}{n_h}
\sum_{i:\,hour_i=h} Y_i.
\
$$
If $(hat\lambda_h)$ varies substantially across $(h)$, a homogeneous Poisson process is inappropriate.


In [5]:

hourly_profile = (
    df.groupby(["hr", "day_type"], as_index=False)["cnt"]
      .agg(mean="mean", median="median", std="std", n="size")
)

p = figure(
    width=900, height=420,
    title="Average hourly bike-rental demand",
    x_axis_label="Hour of day",
    y_axis_label="Mean rentals per hour",
)

for label, sub in hourly_profile.groupby("day_type"):
    p.line(sub["hr"], sub["mean"], line_width=3, legend_label=label)
    p.scatter(sub["hr"], sub["mean"], size=6, legend_label=label)

p.legend.location = "top_left"
p.add_tools(HoverTool(tooltips=[("hour", "$x"), ("mean demand", "$y{0.0}")]))
show(p)



### What should you infer?

On the real data you should normally see strong hourly structure, especially commuting peaks on working days.

That tells us:
$$
\
\lambda \neq \text{constant}.
\
$$
A more realistic model is
$$
\
Y_h \sim \operatorname{Poisson}(\lambda_h),
\
$$
or, viewed in continuous time, a **piecewise non-homogeneous Poisson process (NHPP)** whose intensity changes by hour.

This is a recurring simulation lesson:

> **EDA is not separate from simulation. EDA tells you what the simulator must preserve.**



## 4. Poisson diagnostic: mean versus variance

For a Poisson random variable,
$$
\
E[Y] = \operatorname{Var}(Y) = \lambda.
\
$$
Therefore a useful diagnostic is
$$
\
D = \frac{s^2}{\bar y}.
\
$$
- $(D\approx1)$: Poisson variance is plausible.
- $(D>1)$: **overdispersion**.
- $(D<1)$: underdispersion.

We condition on hour and working-day status first; otherwise changing rates themselves inflate the variance.


In [6]:

poisson_diag = (
    df.groupby(["hr", "workingday"])["cnt"]
      .agg(mean="mean", variance="var", n="size")
      .reset_index()
)

poisson_diag["dispersion"] = poisson_diag["variance"] / poisson_diag["mean"]

display(poisson_diag.head(10))

p = figure(
    width=700, height=520,
    title="Poisson diagnostic: conditional mean vs variance",
    x_axis_label="Conditional mean",
    y_axis_label="Conditional variance",
)

p.scatter(
    poisson_diag["mean"],
    poisson_diag["variance"],
    size=8,
    alpha=0.65,
    marker='circle'
)

lo = min(poisson_diag["mean"].min(), poisson_diag["variance"].min())
hi = max(poisson_diag["mean"].max(), poisson_diag["variance"].max())
p.line([lo, hi], [lo, hi], line_dash="dashed", line_width=2, legend_label="Poisson: variance = mean")

show(p)

display(
    poisson_diag["dispersion"]
    .describe()
    .to_frame("dispersion statistic")
)


,hr,workingday,mean,variance,n,dispersion
0,0,0,90.8000,"2,370.9817",230,26.1121
1,0,1,36.7863,598.6411,496,16.2735
2,1,0,69.5087,"1,335.8405",230,19.2183
3,1,1,16.5526,136.4303,494,8.2422
4,2,0,53.1711,775.7724,228,14.5901
5,2,1,8.6838,43.0850,487,4.9615
6,3,0,25.7753,221.8033,227,8.6053
7,3,1,4.9426,11.5809,470,2.3431
8,4,0,8.2643,27.1157,227,3.2811
9,4,1,5.4298,9.7936,470,1.8037


,dispersion statistic
count,48.0000
mean,45.0613
std,29.4238
min,1.8037
25%,20.1617
50%,41.4433
75%,72.9161
max,99.6241



### Interpretation

If many points lie substantially above the diagonal,
$$
\
s^2 \gg \bar y,
\
$$
then a pure Poisson model is too narrow.

Why might bike demand be overdispersed?

Because even after conditioning on hour and working-day status, unobserved factors remain:

- weather;
- month and season;
- special events;
- latent changes in popularity;
- dependence between nearby hours.

This is important:

> A Poisson simulator can reproduce the **mean profile** while still underestimating extreme variability.

Later we will still use Poisson simulation because it teaches the ST3247 machinery cleanly, but we will label it as a **baseline stochastic model**, not absolute truth.



# Part III — Random-number generation from first principles

## 5. A linear congruential generator (LCG)

Modern libraries use much better PRNGs, but the LCG makes the mechanism visible:
$$
\
Z_{n+1} = (aZ_n + c)\bmod m
\
$$
and
$$
\
U_n = \frac{Z_n}{m}.
\
$$
The sequence is deterministic given the seed.  
This is why simulation is **reproducible**.

### Complexity

Generating $(n)$ values:

- Time: $(\Theta(n))$
- Extra memory: $(\Theta(n)$ if all values are stored, $(\Theta(1))$ if streamed.


In [7]:

@dataclass
class LinearCongruentialGenerator:
    seed: int = 7
    a: int = 1664525
    c: int = 1013904223
    m: int = 2**32

    def sample(self, n: int) -> np.ndarray:
        z = int(self.seed)
        out = np.empty(n, dtype=float)

        for i in range(n):
            z = (self.a * z + self.c) % self.m
            out[i] = z / self.m

        return out


lcg = LinearCongruentialGenerator(seed=RANDOM_SEED)
u_lcg = lcg.sample(10_000)

print("Mean should be close to 0.5:", u_lcg.mean())
print("Variance should be close to 1/12 =", 1/12, "observed:", u_lcg.var())


Mean should be close to 0.5: 0.5016448848450556
Variance should be close to 1/12 = 0.08333333333333333 observed: 0.0842323038559978


In [8]:

hist, edges = np.histogram(u_lcg, bins=30, range=(0, 1))

p1 = figure(
    width=850, height=350,
    title="LCG output: histogram",
    x_axis_label="u",
    y_axis_label="frequency",
)
p1.quad(
    top=hist,
    bottom=0,
    left=edges[:-1],
    right=edges[1:],
    alpha=0.7,
)

p2 = figure(
    width=850, height=350,
    title="Serial plot: U_t versus U_{t+1}",
    x_axis_label="U_t",
    y_axis_label="U_{t+1}",
)
p2.scatter(u_lcg[:-1][:2000], u_lcg[1:][:2000], size=4, alpha=0.35,marker='cross')

show(column(p1, p2))



### What are we checking?

A good uniform PRNG should have approximately:
$$
\
E[U]=\frac12,
\qquad
Var(U)=\frac1{12}.
\
$$
The serial plot is an informal dependence diagnostic.

A histogram alone is **not enough**: a deterministic generator can have a beautifully uniform marginal distribution and still have poor serial structure.



## 6. Inverse-transform sampling: exponential waiting times

If arrivals follow a Poisson process with rate \(\lambda\), then inter-arrival times follow
$$
\
T\sim\operatorname{Exponential}(\lambda).
\
$$
The CDF is
$$
\
F(t)=1-e^{-\lambda t}.
\
$$
Let $(U\sim U(0,1))$. Solving $(U=F(T))$ gives
$$
\
T=-\frac{\log(1-U)}{\lambda}.
\
$$
Because $(1-U)$ is also uniform,
$$
\
\boxed{T=-\frac{\log U}{\lambda}}.
\
$$

In [9]:

def inverse_exponential(u: np.ndarray, rate: float) -> np.ndarray:
    if rate <= 0:
        raise ValueError("rate must be positive")
    u = np.clip(u, np.finfo(float).tiny, 1.0)
    return -np.log(u) / rate


rate = 4.0  # expected 4 events per hour
u = rng.random(50_000)
waiting = inverse_exponential(u, rate)

print("Theoretical mean waiting time:", 1/rate)
print("Simulated mean waiting time:  ", waiting.mean())
print("Theoretical variance:", 1/rate**2)
print("Simulated variance:  ", waiting.var())


Theoretical mean waiting time: 0.25
Simulated mean waiting time:   0.24928371098621574
Theoretical variance: 0.0625
Simulated variance:   0.06250396641409942


In [10]:

hist, edges = np.histogram(waiting, bins=60, density=True)
x = np.linspace(0, np.quantile(waiting, 0.995), 400)
pdf = rate * np.exp(-rate * x)

p = figure(
    width=850, height=400,
    title="Inverse-transform exponential sampler: empirical vs theoretical density",
    x_axis_label="Waiting time (hours)",
    y_axis_label="Density",
)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.45, legend_label="Simulation")
p.line(x, pdf, line_width=3, legend_label="Theoretical Exp(rate)")
show(p)



### Inference

If the empirical histogram follows the theoretical density and the empirical moments approach their theoretical values, the transformation is behaving correctly.

This is a classic simulation workflow:

1. derive the sampler mathematically;
2. implement it;
3. validate distributional properties empirically.



# Part IV — Calibrating a stochastic demand model

## 7. Estimate a piecewise hourly intensity

We concentrate on **working days** because commuter structure is relatively stable.

For each hour $(h)$,
$$
\
\hat\lambda_h = \frac{1}{n_h}\sum_i Y_{ih}.
\
$$
A simple piecewise model is then
$$
\
Y_h \mid h \sim \operatorname{Poisson}(\hat\lambda_h).
\
$$
This is a **plug-in simulation model**: parameters are estimated from data and then treated as fixed during a Monte Carlo run.


In [11]:

working = df[df["workingday"] == 1].copy()

lambda_by_hour = (
    working.groupby("hr")["cnt"]
           .mean()
           .reindex(range(24))
)

if lambda_by_hour.isna().any():
    raise ValueError("Some hours have no observations; cannot construct a 24-hour profile.")

lambda_vec = lambda_by_hour.to_numpy()

display(
    pd.DataFrame({
        "hour": np.arange(24),
        "lambda_hat": lambda_vec,
    })
)


,hour,lambda_hat
0,0,36.7863
1,1,16.5526
2,2,8.6838
3,3,4.9426
4,4,5.4298
5,5,24.9131
6,6,102.5000
7,7,290.6129
8,8,477.0060
9,9,241.5181



## 8. A reusable demand simulator

Conditional on the fitted profile:
$$
\
Y_h^{(b)}\sim \operatorname{Poisson}(\hat\lambda_h),
\quad h=0,\ldots,23.
\
$$
For $(B)$ simulated days we generate a $(B\times24)$ matrix.

### Complexity

For $(B)$ days and $(H=24)$ hours:

- Time: $(\Theta(BH))$
- Memory: $(\Theta(BH))$ if the full matrix is retained.
- If only totals are needed, simulation can be streamed with $(\Theta(B))$ or even $(\Theta(1))$ extra memory.


In [12]:

@dataclass
class PiecewisePoissonDemandModel:
    lambda_by_hour: np.ndarray

    def __post_init__(self):
        self.lambda_by_hour = np.asarray(self.lambda_by_hour, dtype=float)
        if self.lambda_by_hour.shape != (24,):
            raise ValueError("Expected exactly 24 hourly intensities.")
        if np.any(self.lambda_by_hour < 0):
            raise ValueError("Poisson intensities must be non-negative.")

    @property
    def expected_daily_total(self) -> float:
        return float(self.lambda_by_hour.sum())

    def simulate_days(
        self,
        n_days: int,
        random_state: np.random.Generator,
    ) -> np.ndarray:
        return random_state.poisson(
            lam=self.lambda_by_hour,
            size=(n_days, 24),
        )


demand_model = PiecewisePoissonDemandModel(lambda_vec)
sim_days = demand_model.simulate_days(20_000, rng)

print("Theoretical expected daily total:", demand_model.expected_daily_total)
print("Simulated mean daily total:", sim_days.sum(axis=1).mean())


Theoretical expected daily total: 4608.8420591068825
Simulated mean daily total: 4608.8992



## 9. Does the simulator reproduce the hourly mean profile?

For each hour we compare:

- observed mean;
- simulated mean;
- 5th and 95th simulated percentiles.

The mean should match almost automatically because we calibrated \(\lambda_h\) from the same data.

The more interesting question is whether the **spread** is realistic.


In [13]:

sim_mean = sim_days.mean(axis=0)
sim_q05 = np.quantile(sim_days, 0.05, axis=0)
sim_q95 = np.quantile(sim_days, 0.95, axis=0)

profile_source = ColumnDataSource(pd.DataFrame({
    "hour": np.arange(24),
    "observed_mean": lambda_vec,
    "sim_mean": sim_mean,
    "lower": sim_q05,
    "upper": sim_q95,
}))

p = figure(
    width=900, height=430,
    title="Observed hourly mean vs Poisson simulation envelope",
    x_axis_label="Hour",
    y_axis_label="Rentals",
)

band = Band(
    base="hour",
    lower="lower",
    upper="upper",
    source=profile_source,
    fill_alpha=0.18,
    line_alpha=0.0,
)
p.add_layout(band)

p.line("hour", "observed_mean", source=profile_source, line_width=3, legend_label="Observed mean")
p.scatter("hour", "observed_mean", source=profile_source, size=6,marker='circle')
p.line("hour", "sim_mean", source=profile_source, line_width=2, line_dash="dashed", legend_label="Simulated mean")

p.legend.location = "top_left"
show(p)



### Important inference

Matching the conditional mean is **necessary but not sufficient**.

If the real data are overdispersed, the Poisson model typically produces simulation bands that are too narrow.

This separates two concepts:

- **calibration of the mean**;
- **calibration of the entire distribution**.

A good simulator must eventually address both.



# Part V — Non-homogeneous Poisson-process thinking

## 10. From hourly counts to event times

The dataset gives counts, not exact within-hour rental timestamps.

A useful ST3247 exercise is to assume that, conditional on an hourly count \(N_h\), event times are uniformly distributed inside the hour.

Equivalently, under a piecewise-constant NHPP:
$$
\
\lambda(t)=\lambda_h,
\qquad h\le t<h+1.
\
$$
We can simulate each hourly count and then place that many arrivals uniformly within the hour.

This gives a continuous event stream suitable for event-driven simulation.


In [14]:

def simulate_piecewise_nhpp_day(
    lambda_by_hour: np.ndarray,
    random_state: np.random.Generator,
) -> np.ndarray:
    arrivals = []

    for hour, lam in enumerate(lambda_by_hour):
        n = random_state.poisson(lam)
        within_hour = random_state.uniform(0, 1, size=n)
        arrivals.extend(hour + within_hour)

    return np.sort(np.asarray(arrivals, dtype=float))


arrival_times = simulate_piecewise_nhpp_day(lambda_vec, rng)

print("Number of simulated arrivals:", len(arrival_times))
print("First 10 event times (hours after midnight):")
print(arrival_times[:10])


Number of simulated arrivals: 4647
First 10 event times (hours after midnight):
[0.04722523 0.07821339 0.12272454 0.12774238 0.17355582 0.18505722
 0.21655027 0.22429605 0.22676864 0.23125631]


In [15]:

p = figure(
    width=900, height=260,
    title="One simulated NHPP day — individual event times",
    x_axis_label="Hour of day",
    y_axis_label="Event marker",
    y_range=(-0.2, 1.2),
)

# Plot at most 2,000 glyphs for notebook responsiveness.
display_arrivals = arrival_times[:2000]
p.scatter(display_arrivals, np.zeros_like(display_arrivals), size=4, alpha=0.25,marker='circle')
show(p)



### Why this matters

Once you have event times rather than only counts, you can simulate:

- queues;
- server utilisation;
- waiting times;
- congestion;
- inventory depletion;
- failures and repairs.

This is the bridge from **random-variable simulation** to **discrete-event simulation**.



# Part VI — Monte Carlo estimation

## 11. Decision question: how often does daily demand exceed a high threshold?

Let
$$
\
D = \sum_{h=0}^{23}Y_h.
\
$$
Suppose we choose a threshold \(c\). We want
$$
\
p=P(D>c).
\
$$
A Monte Carlo estimator is
$$
\
\hat p_B
=
\frac1B
\sum_{b=1}^{B}
I(D_b>c).
\
$$

For an indicator variable,
$$
\
SE(\hat p_B)
\approx
\sqrt{\frac{\hat p_B(1-\hat p_B)}{B}}.
\
$$
We choose the threshold from the **observed working-day daily totals**, making the question data-driven.


In [16]:

observed_daily = (
    working.groupby("date")["cnt"]
           .sum()
           .sort_index()
)

threshold = float(observed_daily.quantile(0.90))

print("Observed 90th-percentile working-day demand threshold:", threshold)
display(observed_daily.describe().to_frame("observed daily demand"))


Observed 90th-percentile working-day demand threshold: 7335.3


,observed daily demand
count,500.0000
mean,"4,584.8200"
std,"1,878.4156"
min,22.0000
25%,"3,344.2500"
50%,"4,582.0000"
75%,"5,987.5000"
max,"8,362.0000"


In [17]:

@dataclass
class MonteCarloProbabilityEstimate:
    estimate: float
    standard_error: float
    ci_low: float
    ci_high: float
    n: int

    @classmethod
    def from_boolean(cls, values: np.ndarray) -> "MonteCarloProbabilityEstimate":
        values = np.asarray(values, dtype=float)
        n = len(values)
        p = values.mean()
        se = math.sqrt(max(p * (1 - p), 0.0) / n)
        z = stats.norm.ppf(0.975)
        return cls(
            estimate=float(p),
            standard_error=float(se),
            ci_low=float(max(0, p - z * se)),
            ci_high=float(min(1, p + z * se)),
            n=n,
        )


B = 100_000
mc_sim = demand_model.simulate_days(B, rng)
mc_totals = mc_sim.sum(axis=1)
exceeded = mc_totals > threshold

mc_result = MonteCarloProbabilityEstimate.from_boolean(exceeded)
mc_result


MonteCarloProbabilityEstimate(estimate=0.0, standard_error=0.0, ci_low=0.0, ci_high=0.0, n=100000)


### Interpretation

The confidence interval above measures **Monte Carlo error conditional on the fitted model**.

It does **not** include uncertainty about:

- the estimated $(lambda_h)$;
- whether Poisson is the correct model;
- how future demand differs from 2011–2012.

That distinction is crucial.
$$
\
\boxed{
\text{Monte Carlo uncertainty} \neq \text{model uncertainty}
}
\
$$


## 12. Monte Carlo convergence

Because
$$
\
SE(\hat\theta_B)=O(B^{-1/2}),
\
$$
quadrupling the number of simulations approximately halves the standard error.

We visualize the running estimate of $(P(D>c))$.


In [18]:

checkpoints = np.unique(
    np.logspace(2, np.log10(B), 70).astype(int)
)

running_p = np.array([
    exceeded[:n].mean()
    for n in checkpoints
])

running_se = np.sqrt(
    np.clip(running_p * (1-running_p), 0, None) / checkpoints
)

p = figure(
    width=900, height=420,
    x_axis_type="log",
    title="Monte Carlo convergence of exceedance probability",
    x_axis_label="Number of simulated days B (log scale)",
    y_axis_label="Estimated probability",
)

p.line(checkpoints, running_p, line_width=3)
p.line(checkpoints, running_p + 1.96*running_se, line_dash="dashed")
p.line(checkpoints, running_p - 1.96*running_se, line_dash="dashed")

show(p)



### Computational lesson

Brute-force improvement is expensive:
$$
\
SE \propto B^{-1/2}.
\
$$
To reduce standard error by a factor of 10, we typically need roughly 100 times more simulation.

That motivates **variance reduction**.



# Part VII — Variance reduction using a control variate

## 13. Why a control variate works

Our target is
$$
\
H = I(D>c).
\
$$
The daily total \(D\) is strongly associated with \(H\), and under our fitted Poisson model its expectation is known:
$$
\
E[D]
=
\sum_{h=0}^{23}\lambda_h
=
\mu_D.
\
$$
Define
$$
\
H_{CV}
=
H-\beta(D-\mu_D).
\
$$
Because
$$
\
E[D-\mu_D]=0,
\
$$
the expectation is unchanged:
$$
\
E[H_{CV}] = E[H].
\
$$
The variance-minimising coefficient is
$$
\
\beta^\star
=
\frac{\operatorname{Cov}(H,D)}
{\operatorname{Var}(D)}.
\
$$

In [19]:


def control_variate_probability(
    totals: np.ndarray,
    threshold: float,
    known_mean_total: float,
) -> dict[str, float]:
    totals = np.asarray(totals, dtype=float)
    h = (totals > threshold).astype(float)

    # Compute covariance and variance
    cov_hd = np.cov(h, totals, ddof=1)[0, 1]
    var_d = np.var(totals, ddof=1)

    # Guard against division by zero
    beta = cov_hd / var_d if var_d != 0 else np.nan

    # Adjusted estimator
    adjusted = h - beta * (totals - known_mean_total)

    # Estimates
    naive_est = h.mean()
    cv_est = adjusted.mean()

    # Standard errors
    naive_se = h.std(ddof=1) / np.sqrt(len(h))
    cv_se = adjusted.std(ddof=1) / np.sqrt(len(adjusted))

    # Variance reduction factor with safe handling
    if cv_se == 0 or np.isnan(cv_se):
        variance_reduction_factor = np.nan
    else:
        variance_reduction_factor = (naive_se / cv_se) ** 2

    return {
        "naive_estimate": naive_est,
        "control_variate_estimate": cv_est,
        "beta": beta,
        "naive_se": naive_se,
        "control_variate_se": cv_se,
        "variance_reduction_factor": variance_reduction_factor,
    }


# Example usage
cv_result = control_variate_probability(
    totals=mc_totals,
    threshold=threshold,
    known_mean_total=demand_model.expected_daily_total,
)

display(pd.Series(cv_result).to_frame("value"))


,value
naive_estimate,0.0000
control_variate_estimate,0.0000
beta,0.0000
naive_se,0.0000
control_variate_se,0.0000
variance_reduction_factor,NaN



### How to read the efficiency result

If the variance-reduction factor is, say,
$$
\
R=4,
\
$$
then the control-variate estimator gives approximately the same variance as a naive Monte Carlo run using **four times as many samples**.

That is why variance reduction is algorithmically valuable:

> Instead of buying accuracy only with more CPU, exploit mathematical structure.



# Part VIII — Bootstrap: sampling uncertainty is different from Monte Carlo uncertainty

## 14. Bootstrap the empirical mean daily demand

Now we temporarily stop assuming the fitted Poisson model.

Let the observed working-day totals be
$$
\
d_1,\ldots,d_n.
\
$$
A non-parametric bootstrap:

1. resamples $(n)$ observed days with replacement;
2. computes the mean;
3. repeats the procedure.

This approximates **sampling uncertainty in the empirical mean**.

Contrast:

- Monte Carlo SE: uncertainty because we simulated only finitely many runs from a model.
- Bootstrap SE: uncertainty because we observed only finitely many real days.


In [20]:

def bootstrap_mean(
    x: np.ndarray,
    n_boot: int,
    random_state: np.random.Generator,
) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = len(x)

    indices = random_state.integers(0, n, size=(n_boot, n))
    return x[indices].mean(axis=1)


boot_means = bootstrap_mean(
    observed_daily.to_numpy(),
    n_boot=10_000,
    random_state=rng,
)

bootstrap_ci = np.quantile(boot_means, [0.025, 0.975])

print("Observed mean daily demand:", observed_daily.mean())
print("Bootstrap 95% percentile CI:", bootstrap_ci)
print("Bootstrap SE:", boot_means.std(ddof=1))


Observed mean daily demand: 4584.82
Bootstrap 95% percentile CI: [4421.935   4750.94405]
Bootstrap SE: 84.39180891593763


In [21]:

hist, edges = np.histogram(boot_means, bins=50, density=True)

p = figure(
    width=850, height=400,
    title="Bootstrap distribution of mean working-day demand",
    x_axis_label="Bootstrap mean",
    y_axis_label="Density",
)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.55)
show(p)



### Caveat: time dependence

The simple bootstrap above resamples days as if they were IID.

Bike demand has:

- seasonal structure;
- long-term growth;
- weather dependence.

Therefore a time-series-aware analysis might use:

- stratified bootstrap;
- moving-block bootstrap;
- model-based bootstrap.

The simple IID bootstrap is used here because the goal is to teach the simulation mechanism first.



# Part IX — Discrete-event simulation

## 15. A hypothetical station-level queue

The original dataset contains **system-wide rentals**, not station-level queues.

For a pedagogical DES example, suppose a particular station receives a fraction
$$
\
q=0.02
\
$$
of system demand.

Then its hourly rate is
$$
\
\lambda_h^{station}=q\lambda_h.
\
$$
Assume each rental-processing interaction has an exponential service time with mean 2 minutes.

We will simulate $(c)$ parallel service channels.

This is an explicit **scenario assumption**—not an observed property of the data.


In [22]:

@dataclass
class QueueSimulationResult:
    n_arrivals: int
    mean_wait_minutes: float
    p95_wait_minutes: float
    probability_wait: float
    max_wait_minutes: float


@dataclass
class MultiServerQueueSimulator:
    service_mean_minutes: float = 2.0

    def simulate(
        self,
        arrival_times_hours: np.ndarray,
        n_servers: int,
        random_state: np.random.Generator,
    ) -> QueueSimulationResult:
        if n_servers <= 0:
            raise ValueError("n_servers must be >= 1")

        if len(arrival_times_hours) == 0:
            return QueueSimulationResult(0, 0, 0, 0, 0)

        # Min-heap of server next-available times, in hours.
        server_heap = [0.0] * n_servers
        heapq.heapify(server_heap)

        waits = []

        for arrival in arrival_times_hours:
            available = heapq.heappop(server_heap)

            service_start = max(arrival, available)
            wait = service_start - arrival

            service_hours = random_state.exponential(
                self.service_mean_minutes / 60.0
            )
            departure = service_start + service_hours

            heapq.heappush(server_heap, departure)
            waits.append(wait)

        waits = np.asarray(waits) * 60.0

        return QueueSimulationResult(
            n_arrivals=len(waits),
            mean_wait_minutes=float(waits.mean()),
            p95_wait_minutes=float(np.quantile(waits, 0.95)),
            probability_wait=float(np.mean(waits > 1e-12)),
            max_wait_minutes=float(waits.max()),
        )


station_share = 0.02
station_lambda = lambda_vec * station_share

queue_simulator = MultiServerQueueSimulator(service_mean_minutes=2.0)

rows = []
for servers in [1, 2, 3]:
    day_arrivals = simulate_piecewise_nhpp_day(station_lambda, rng)
    result = queue_simulator.simulate(day_arrivals, servers, rng)
    rows.append({"servers": servers, **result.__dict__})

queue_comparison = pd.DataFrame(rows)
display(queue_comparison)


,servers,n_arrivals,mean_wait_minutes,p95_wait_minutes,probability_wait,max_wait_minutes
0,1,82,0.3455,3.1370,0.1829,4.6307
1,2,77,0.0000,0.0000,0.0000,0.0000
2,3,91,0.0000,0.0000,0.0000,0.0000



### DES algorithmic structure

For every arrival:

1. retrieve the earliest available server;
2. compute service start time;
3. calculate waiting time;
4. draw a service duration;
5. update that server's availability.

A min-heap makes server selection efficient.

If there are $(A)$ arrivals and $(c)$ servers:

- each heap pop/push costs $(O(\log c))$;
- total time is $(O(A\log c))$;
- memory for server state is $(O(c))$, plus whatever output statistics are retained.

For a small fixed number of servers, this is practically close to linear in \(A\).



## 16. Replication: never make a decision from one simulated day

One DES trajectory can be lucky or unlucky.

We therefore perform many independent replications and estimate:
$$
\
E[W],
\qquad
P(W>0),
\qquad
Q_{0.95}(W).
\
$$
This is one of the most important habits in simulation studies.


In [23]:

def replicate_queue_policy(
    station_lambda: np.ndarray,
    servers: int,
    n_replications: int,
    queue_simulator: MultiServerQueueSimulator,
    random_state: np.random.Generator,
) -> pd.DataFrame:
    rows = []

    for rep in range(n_replications):
        arrivals = simulate_piecewise_nhpp_day(station_lambda, random_state)
        result = queue_simulator.simulate(arrivals, servers, random_state)
        rows.append(result.__dict__)

    return pd.DataFrame(rows)


queue_policy_results = []

for servers in [1, 2, 3]:
    reps = replicate_queue_policy(
        station_lambda=station_lambda,
        servers=servers,
        n_replications=1000,
        queue_simulator=queue_simulator,
        random_state=rng,
    )

    queue_policy_results.append({
        "servers": servers,
        "mean_of_mean_wait": reps["mean_wait_minutes"].mean(),
        "mean_probability_wait": reps["probability_wait"].mean(),
        "mean_p95_wait": reps["p95_wait_minutes"].mean(),
        "p95_of_daily_mean_wait": reps["mean_wait_minutes"].quantile(0.95),
    })

queue_policy_summary = pd.DataFrame(queue_policy_results)
display(queue_policy_summary)


,servers,mean_of_mean_wait,mean_probability_wait,mean_p95_wait,p95_of_daily_mean_wait
0,1,0.5480,0.2018,3.6182,1.1230
1,2,0.0250,0.0216,0.0351,0.0930
2,3,0.0009,0.0016,0.0000,0.0064


In [24]:

p = figure(
    width=750, height=400,
    title="Queue policy comparison across Monte Carlo replications",
    x_axis_label="Number of service channels",
    y_axis_label="Expected mean wait (minutes)",
)

p.line(
    queue_policy_summary["servers"],
    queue_policy_summary["mean_of_mean_wait"],
    line_width=3,
)
p.circle(
    queue_policy_summary["servers"],
    queue_policy_summary["mean_of_mean_wait"],
    size=9,
)
show(p)



### Decision interpretation

The simulation lets us ask a prescriptive question:

> How much waiting-time reduction do we buy by adding another service channel?

But do not overclaim.  
Our conclusion is conditional on:

- station share \(q\);
- hourly demand model;
- exponential service times;
- independence assumptions.

A professional simulation report always separates **data-backed inputs** from **scenario assumptions**.



# Part X — MCMC: simulate from a posterior distribution

## 17. Bayesian Poisson-rate example

Take one relatively homogeneous slice—for example:

- working days;
- hour 08:00;
- clear/partly cloudy conditions.

Model:

\[
Y_i\mid\lambda\sim Poisson(\lambda).
\]

Use a Gamma prior in **shape-rate** form:

\[
\lambda\sim Gamma(a_0,b_0).
\]

Because Gamma is conjugate to Poisson, the exact posterior is

\[
\lambda\mid y
\sim
Gamma\left(
a_0+\sum_i y_i,\;
b_0+n
\right).
\]

Why run MCMC when the exact answer exists?

Because it gives us a **ground-truth benchmark** for validating a Metropolis–Hastings implementation.


In [25]:

slice_counts = df.loc[
    (df["workingday"] == 1)
    & (df["hr"] == 8)
    & (df["weathersit"] == 1),
    "cnt",
].to_numpy(dtype=int)

# If an unusually small slice occurs in a modified dataset, relax weather filtering.
if len(slice_counts) < 30:
    slice_counts = df.loc[
        (df["workingday"] == 1) & (df["hr"] == 8),
        "cnt",
    ].to_numpy(dtype=int)

print("Number of observations:", len(slice_counts))
print("Sample mean:", slice_counts.mean())
print("Sample variance:", slice_counts.var(ddof=1))


Number of observations: 280
Sample mean: 503.8642857142857
Sample variance: 30713.580081925244



## 18. Metropolis–Hastings on log-rate

To guarantee $(\lambda>0)$, work with
$$
\
\eta=\log\lambda.
\
$$
Proposal:
$$
\
\eta' = \eta+\epsilon,
\qquad
\epsilon\sim N(0,\sigma^2).
\
$$
Because our Markov chain is expressed in \(\eta\), the log-target includes the Jacobian:
$$
\
\log p_\eta(\eta\mid y)
=
\log p_\lambda(e^\eta\mid y)+\eta.
\
$$
This is a subtle but important implementation detail.


In [26]:

@dataclass
class PoissonGammaMCMC:
    prior_shape: float = 2.0
    prior_rate: float = 0.02
    proposal_sd: float = 0.08

    def log_posterior_eta(
        self,
        eta: float,
        y_sum: int,
        n: int,
    ) -> float:
        lam = math.exp(eta)

        # Constants not involving lambda are omitted.
        log_likelihood = y_sum * eta - n * lam
        log_prior = (
            (self.prior_shape - 1) * eta
            - self.prior_rate * lam
        )

        # Jacobian for lambda = exp(eta)
        log_jacobian = eta

        return log_likelihood + log_prior + log_jacobian

    def sample(
        self,
        y: np.ndarray,
        n_steps: int,
        random_state: np.random.Generator,
    ) -> tuple[np.ndarray, float]:
        y = np.asarray(y)
        y_sum = int(y.sum())
        n = len(y)

        eta = math.log(max(y.mean(), 1e-6))
        current_lp = self.log_posterior_eta(eta, y_sum, n)

        samples = np.empty(n_steps)
        accepted = 0

        for t in range(n_steps):
            proposal = eta + random_state.normal(0, self.proposal_sd)
            proposal_lp = self.log_posterior_eta(proposal, y_sum, n)

            log_alpha = proposal_lp - current_lp

            if math.log(random_state.random()) < min(0.0, log_alpha):
                eta = proposal
                current_lp = proposal_lp
                accepted += 1

            samples[t] = math.exp(eta)

        return samples, accepted / n_steps


mcmc = PoissonGammaMCMC(
    prior_shape=2.0,
    prior_rate=0.02,
    proposal_sd=0.08,
)

chain, acceptance_rate = mcmc.sample(
    slice_counts,
    n_steps=25_000,
    random_state=rng,
)

burn = 5_000
posterior_draws = chain[burn:]

print("MH acceptance rate:", acceptance_rate)
print("Posterior mean from MCMC:", posterior_draws.mean())


MH acceptance rate: 0.04232
Posterior mean from MCMC: 503.7897151794903



### Acceptance-rate interpretation

There is no universal perfect acceptance rate.

Very roughly:

- extremely low acceptance → proposals are often too large;
- extremely high acceptance → proposals may be too small, causing slow exploration;
- the real target is **good mixing and effective exploration**, not an arbitrary acceptance percentage.

Always inspect the trace and autocorrelation as well.


In [27]:

a0 = mcmc.prior_shape
b0 = mcmc.prior_rate

post_shape = a0 + slice_counts.sum()
post_rate = b0 + len(slice_counts)
post_scale = 1 / post_rate

exact_mean = post_shape / post_rate
exact_sd = math.sqrt(post_shape) / post_rate

print("Exact posterior mean:", exact_mean)
print("Exact posterior SD:", exact_sd)
print("MCMC posterior mean:", posterior_draws.mean())
print("MCMC posterior SD:", posterior_draws.std(ddof=1))


Exact posterior mean: 503.8354403256911
Exact posterior SD: 1.3413738355165665
MCMC posterior mean: 503.7897151794903
MCMC posterior SD: 1.394251592452446


In [28]:

# Trace plot
trace_index = np.arange(len(chain))

p1 = figure(
    width=900, height=320,
    title="Metropolis–Hastings trace for Poisson rate λ",
    x_axis_label="Iteration",
    y_axis_label="λ",
)
p1.line(trace_index, chain, line_width=1, alpha=0.6)

# Posterior histogram vs exact Gamma posterior
hist, edges = np.histogram(posterior_draws, bins=60, density=True)
x = np.linspace(
    np.quantile(posterior_draws, 0.001),
    np.quantile(posterior_draws, 0.999),
    500,
)
exact_pdf = stats.gamma.pdf(
    x,
    a=post_shape,
    scale=post_scale,
)

p2 = figure(
    width=900, height=370,
    title="MCMC posterior vs exact Gamma posterior",
    x_axis_label="λ",
    y_axis_label="Density",
)
p2.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.45, legend_label="MCMC")
p2.line(x, exact_pdf, line_width=3, legend_label="Exact posterior")

show(column(p1, p2))



## 19. Simple autocorrelation diagnostic

For lag \(k\),

\[
\rho_k
=
Corr(\lambda_t,\lambda_{t+k}).
\]

IID Monte Carlo samples ideally have negligible serial correlation.

MCMC samples are *supposed* to be dependent, but very high autocorrelation means that a long chain contains much less information than the raw iteration count suggests.


In [29]:

def autocorrelation(x: np.ndarray, max_lag: int = 40) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.dot(x, x)

    return np.array([
        1.0 if lag == 0 else np.dot(x[:-lag], x[lag:]) / denom
        for lag in range(max_lag + 1)
    ])


acf = autocorrelation(posterior_draws, max_lag=40)

p = figure(
    width=850, height=350,
    title="MCMC autocorrelation",
    x_axis_label="Lag",
    y_axis_label="Autocorrelation",
)
p.vbar(x=np.arange(len(acf)), top=acf, width=0.7)
show(p)



### MCMC complexity insight

A naive Poisson log-likelihood calculation would revisit all \(n\) observations at every MCMC step:
$$
\
\Theta(Tn).
\
$$
But the Poisson likelihood depends on the data through the sufficient statistic
$$
\
S=\sum_iY_i.
\
$$
Precompute \(S\) once in \(\Theta(n)\), then each Metropolis step is \(\Theta(1)\):
$$
\
\boxed{\Theta(n+T)}
\
$$
instead of
$$
\
\Theta(Tn).
\
$$
That is a powerful example of **statistics improving computational complexity**.



# Part XI — Model criticism: improve the baseline

## 20. Overdispersion suggests a Gamma–Poisson / Negative-Binomial extension

Suppose the hourly rate itself fluctuates:
$$
\
\Lambda_h\sim Gamma(\alpha,\beta),
\
$$
and
$$
\
Y_h\mid\Lambda_h\sim Poisson(\Lambda_h).
\
$$
Marginally, \(Y_h\) follows a Negative Binomial distribution and can have
$$
\
Var(Y_h)>E[Y_h].
\
$$
This is one principled way to model the overdispersion we diagnosed earlier.

We estimate a simple method-of-moments Negative-Binomial approximation separately by hour:
$$
\
Var(Y)
=
\mu+\frac{\mu^2}{r}.
\
$$
Therefore,
$$
\
r
=
\frac{\mu^2}{Var(Y)-\mu}
\
$$
when $(Var(Y)>\mu)$.


In [30]:

def fit_nb_moments(group: pd.Series) -> tuple[float, float]:
    mu = float(group.mean())
    var = float(group.var(ddof=1))

    if var <= mu:
        # Large r approximates Poisson.
        return mu, 1e9

    r = mu**2 / (var - mu)
    return mu, r


nb_params = []
for hour in range(24):
    y = working.loc[working["hr"] == hour, "cnt"]
    mu, r = fit_nb_moments(y)
    nb_params.append((hour, mu, r))

nb_params = pd.DataFrame(nb_params, columns=["hour", "mu", "r"])
display(nb_params.head())


,hour,mu,r
0,0,36.7863,2.4085
1,1,16.5526,2.2856
2,2,8.6838,2.1920
3,3,4.9426,3.6799
4,4,5.4298,6.7562



For NumPy's parameterisation,
$$
\
Y\sim NB(r,p),
\
$$
with
$$
\
E[Y]=r\frac{1-p}{p}.
\
$$
So given \(\mu\) and \(r\),
$$
\
p=\frac{r}{r+\mu}.
\
$$

In [31]:

def simulate_nb_days(
    params: pd.DataFrame,
    n_days: int,
    random_state: np.random.Generator,
) -> np.ndarray:
    out = np.empty((n_days, 24), dtype=int)

    for row in params.itertuples(index=False):
        h = int(row.hour)
        mu = float(row.mu)
        r = float(row.r)

        if r > 1e8:
            out[:, h] = random_state.poisson(mu, size=n_days)
        else:
            p = r / (r + mu)
            out[:, h] = random_state.negative_binomial(r, p, size=n_days)

    return out


nb_days = simulate_nb_days(nb_params, 30_000, rng)

poisson_daily_totals = demand_model.simulate_days(30_000, rng).sum(axis=1)
nb_daily_totals = nb_days.sum(axis=1)

comparison = pd.DataFrame({
    "Observed": [
        observed_daily.mean(),
        observed_daily.var(ddof=1),
        observed_daily.quantile(0.95),
    ],
    "Poisson simulation": [
        poisson_daily_totals.mean(),
        poisson_daily_totals.var(ddof=1),
        np.quantile(poisson_daily_totals, 0.95),
    ],
    "NB simulation": [
        nb_daily_totals.mean(),
        nb_daily_totals.var(ddof=1),
        np.quantile(nb_daily_totals, 0.95),
    ],
}, index=["mean", "variance", "95th percentile"])

display(comparison)


,Observed,Poisson simulation,NB simulation
mean,"4,584.8200","4,608.4343","4,610.8015"
variance,"3,528,445.1018","4,631.2823","280,664.5206"
95th percentile,"7,539.6000","4,721.0000","5,519.0000"



### Key modelling insight

A richer simulator may reproduce tail behaviour better, but that does not automatically make it *correct*.

The observed daily variability also contains:

- serial dependence between hours;
- day-to-day weather effects;
- seasonality;
- annual trend.

A fully specified simulator should model these structures explicitly rather than attributing all extra variation to an independent Negative-Binomial draw.

This is exactly the mindset simulation courses try to develop:
$$
\
\boxed{
\text{simulate} \rightarrow \text{diagnose} \rightarrow
\text{identify mismatch} \rightarrow \text{refine}
}
\
$$


# Part XII — A compact simulation study

## 21. Compare two stochastic models against observed working-day totals

We compare:

1. **Piecewise Poisson**
2. **Hourly Negative Binomial**

using several summary statistics.

This is not predictive cross-validation; it is **simulation validation**.

A simulator should reproduce whichever empirical features are relevant to the decision problem.


In [32]:

def summary_stats(x: np.ndarray) -> dict[str, float]:
    x = np.asarray(x, dtype=float)
    return {
        "mean": x.mean(),
        "sd": x.std(ddof=1),
        "q05": np.quantile(x, 0.05),
        "median": np.quantile(x, 0.50),
        "q95": np.quantile(x, 0.95),
        "max": x.max(),
    }


validation_table = pd.DataFrame({
    "Observed working days": summary_stats(observed_daily.to_numpy()),
    "Piecewise Poisson": summary_stats(poisson_daily_totals),
    "Hourly Negative Binomial": summary_stats(nb_daily_totals),
}).T

display(validation_table)


,mean,sd,q05,median,q95,max
Observed working days,"4,584.8200","1,878.4156","1,525.2000","4,582.0000","7,539.6000","8,362.0000"
Piecewise Poisson,"4,608.4343",68.0535,"4,497.9500","4,608.0000","4,721.0000","4,918.0000"
Hourly Negative Binomial,"4,610.8015",529.7778,"3,783.0000","4,586.0000","5,519.0000","7,344.0000"


In [33]:

# ECDF visualization
def ecdf(x: np.ndarray):
    x = np.sort(np.asarray(x))
    y = np.arange(1, len(x)+1) / len(x)
    return x, y


p = figure(
    width=900, height=430,
    title="Empirical CDF: observed vs simulated daily totals",
    x_axis_label="Daily rentals",
    y_axis_label="CDF",
)

for name, arr in {
    "Observed": observed_daily.to_numpy(),
    "Poisson": poisson_daily_totals,
    "Negative Binomial": nb_daily_totals,
}.items():
    x, y = ecdf(arr)
    # Downsample huge curves only for rendering efficiency.
    step = max(1, len(x) // 3000)
    p.line(x[::step], y[::step], line_width=2.5, legend_label=name)

p.legend.location = "bottom_right"
show(p)



### How to choose between simulation models

Do **not** simply choose the model with the prettiest plot.

Ask what quantity your downstream decision depends on.

For example:

- staffing → upper-tail demand and waiting-time distributions;
- average resource use → mean and time profile;
- extreme-risk planning → high quantiles and tail dependence;
- inference → calibrated likelihood/posterior behaviour.

Model validation must be aligned with the **decision functional**.



# Part XIII — Exercises with guided answers

## Exercise 1 — Why is a homogeneous Poisson process unsuitable?

Try to answer before expanding mentally:

A homogeneous Poisson process requires constant \(\lambda\), whereas the hourly profile shows strong systematic variation across the day. Therefore a constant-rate model confounds predictable time-of-day structure with randomness.

---

## Exercise 2 — Why can the Poisson model match the mean but miss the tails?

Because Poisson constrains
$$
\
Var(Y)=E[Y].
\
$$
If the data are overdispersed, fitting \(\lambda\) to the mean cannot independently increase the variance.

---

## Exercise 3 — What uncertainty does the Monte Carlo CI *not* include?

It does not account for uncertainty in fitted parameters or model misspecification. It only reflects finite simulation error conditional on the chosen stochastic model.

---

## Exercise 4 — Why is the control variate unbiased?

Because
$$
\
E[D-\mu_D]=0.
\
$$
Hence subtracting
$$
\
\beta(D-\mu_D)
\
$$
changes variance but not expectation.

---

## Exercise 5 — Why can sufficient statistics accelerate MCMC?

For a Poisson sample, the likelihood's dependence on the observations can be compressed into
$$
\
\sum_iY_i.
\
$$
So we avoid repeatedly scanning the full dataset for every proposal.

---

## Exercise 6 — Extend the NHPP

Fit separate rate profiles
$$
\
\lambda_{h,w}
\
$$
for weather state \(w\), and simulate weather first, then arrivals conditional on weather.

This introduces **hierarchical simulation**:
$$
\
W\rightarrow\Lambda\rightarrow Y.
\
$$
---

## Exercise 7 — Add common random numbers

Compare two queue policies using exactly the same arrival streams and, where appropriate, the same service-time random numbers. This induces positive correlation between policy outcomes and can reduce the variance of the *difference* between policies.



# Part XIV — Complexity and engineering summary

| Component | Time complexity | Main memory | Engineering observation |
|---|---:|---:|---|
| LCG generation | \(\Theta(n)\) | \(\Theta(n)\) stored / \(\Theta(1)\) streamed | PRNG is deterministic and reproducible |
| inverse exponential transform | \(\Theta(n)\) | \(\Theta(n)\) | vectorisable |
| hourly Poisson Monte Carlo | \(\Theta(BH)\) | \(\Theta(BH)\) | batch/stream if \(B\) is huge |
| control variate | \(\Theta(B)\) | \(\Theta(B)\) | buys accuracy without proportional compute |
| bootstrap mean | \(\Theta(Bn)\) | potentially \(\Theta(Bn)\) | batch the index matrix for large datasets |
| NHPP event generation | \(\Theta(A)\) plus sorting | \(\Theta(A)\) | avoid sorting if generated hour-by-hour already ordered |
| multi-server DES | \(O(A\log c)\) | \(O(c)+O(A)\) outputs | heap is natural for next-available server |
| naive MCMC likelihood | \(\Theta(Tn)\) | \(\Theta(T)\) | often unnecessary |
| sufficient-statistic MCMC | \(\Theta(n+T)\) | \(\Theta(T)\) | exploit probabilistic structure |

where:

- \(B\): Monte Carlo replications;
- \(H=24\): hours per day;
- \(A\): arrivals;
- \(c\): servers;
- \(T\): MCMC steps;
- \(n\): observed sample size.

### Efficiency tips

1. **Vectorise independent Monte Carlo draws** with NumPy.
2. **Stream** when only running means/variances are required.
3. Use **Welford's algorithm** for stable online variance.
4. Use **sufficient statistics** before expensive repeated likelihood evaluation.
5. Use **variance reduction** before blindly multiplying simulations.
6. Keep simulation randomness reproducible with an explicit `Generator`.
7. For policy comparison, use **common random numbers** when valid.
8. Separate:
   - data uncertainty,
   - parameter uncertainty,
   - model uncertainty,
   - Monte Carlo uncertainty.



# Part XV — Final conceptual map

This case study can be summarized as:
$$
\
\text{Observed event counts}
\
$$
$$
\
\downarrow
\
$$
$$
\
\text{EDA + stochastic assumptions}
\
$$
$$
\
\downarrow
\
$$
$$
\
\lambda_h
\
$$
$$
\
\downarrow
\
$$
$$
\
\text{Poisson / Negative-Binomial simulation}
\
$$
$$
\
\downarrow
\
$$
$$
\
\text{NHPP event times}
\
$$
$$
\
\downarrow
\
$$
$$
\
\text{Monte Carlo estimation}
\
$$
$$
\
\downarrow
\
$$
$$
\
\text{variance reduction}
\
$$
$$
\
\downarrow
\
$$
$$
\
\text{discrete-event system simulation}
\
$$
and separately:
$$
\
\text{Observed counts}
\rightarrow
\text{Bayesian model}
\rightarrow
\text{MCMC}
\rightarrow
\text{posterior simulation}.
\
$$
The main ST3247 lesson is not merely “generate random numbers.”

It is:

> **Use mathematically controlled randomness as a computational tool for estimation, inference, system experimentation, and decision-making—then quantify how much uncertainty the simulation itself introduces.**

---

## Suggested next experiments

For deeper study, try adding:

- rejection sampling from scratch;
- alias sampling for discrete weather states;
- antithetic variates;
- importance sampling for rare high-demand events;
- block bootstrap for time-series dependence;
- Gamma-Poisson hierarchical simulation;
- MCMC effective sample size;
- simulated annealing for capacity optimisation;
- an SIR stochastic epidemic simulator;
- Approximate Bayesian Computation (ABC).

Those extensions move very naturally from ST3247 into advanced computational statistics.
